# PipelineWatch-NG — Module 4: Multi-Source Fusion & Risk Scoring
**Goal:** Fuse outputs from all three detectors (SAR spills, AIS vessels, FIRMS fires) onto a common geospatial grid. Compute a combined **Pipeline Risk Score** per grid cell. Generate alert-level classifications (LOW / MEDIUM / HIGH / CRITICAL) ready for the dashboard.


## 4.1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, warnings, numpy as np, pandas as pd
import geopandas as gpd, folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from shapely.geometry import Point, box, Polygon
warnings.filterwarnings('ignore')

!pip install geopandas folium shapely scipy -q

PROJECT_ROOT = '/content/drive/MyDrive/PipelineWatch_NG'
PROCESSED    = f'{PROJECT_ROOT}/data/processed'
MODELS       = f'{PROJECT_ROOT}/models'
OUTPUTS      = f'{PROJECT_ROOT}/outputs'
AOI_BOUNDS   = (5.8, 4.2, 7.2, 5.2)

print('✅ Environment ready.')


## 4.2 — Build the risk grid

In [ ]:
# Define 0.1° × 0.1° grid cells over the AOI (~11km × 11km per cell)
CELL_DEG = 0.1

lon_edges = np.arange(AOI_BOUNDS[0], AOI_BOUNDS[2] + CELL_DEG, CELL_DEG)
lat_edges = np.arange(AOI_BOUNDS[1], AOI_BOUNDS[3] + CELL_DEG, CELL_DEG)

grid_rows = []
for i, lat0 in enumerate(lat_edges[:-1]):
    for j, lon0 in enumerate(lon_edges[:-1]):
        lat1, lon1 = lat0 + CELL_DEG, lon0 + CELL_DEG
        grid_rows.append({
            'cell_id': f'R{i:02d}C{j:02d}',
            'lat0': lat0, 'lon0': lon0,
            'lat1': lat1, 'lon1': lon1,
            'centroid_lat': (lat0 + lat1) / 2,
            'centroid_lon': (lon0 + lon1) / 2,
            'geometry': box(lon0, lat0, lon1, lat1)
        })

gdf_grid = gpd.GeoDataFrame(grid_rows, crs='EPSG:4326')
print(f'✅ Risk grid: {len(gdf_grid)} cells ({len(lat_edges)-1} rows × {len(lon_edges)-1} cols)')
print(f'   Cell size: {CELL_DEG}° × {CELL_DEG}° (~11km × 11km)')


## 4.3 — Load all detector outputs

In [ ]:
import joblib

# ── SAR spill blobs ──────────────────────────────────────────────────────────
blobs_path = f'{PROCESSED}/sar_darkspot_blobs.csv'
if os.path.exists(blobs_path):
    df_blobs = pd.read_csv(blobs_path)
    print(f'SAR blobs loaded: {len(df_blobs)}')
else:
    # Synthetic blobs with approximate coordinates in AOI
    np.random.seed(42)
    df_blobs = pd.DataFrame({
        'blob_id':    range(1, 21),
        'pixel_count':np.random.exponential(120, 20).astype(int) + 20,
        'mean_sigma0':np.random.normal(-23, 2, 20),
        'lat': np.random.uniform(4.3, 5.1, 20),
        'lon': np.random.uniform(5.9, 7.1, 20),
    })
    df_blobs['spill_prob'] = np.random.uniform(0.3, 0.98, 20)
    print(f'⚠️  Using synthetic SAR blobs: {len(df_blobs)}')

# ── AIS anomalies ─────────────────────────────────────────────────────────────
def gen_ais_alerts(n=30):
    np.random.seed(7)
    return pd.DataFrame({
        'mmsi':          [100000000+i for i in range(n)],
        'lat':           np.random.uniform(4.25, 5.15, n),
        'lon':           np.random.uniform(5.85, 7.15, n),
        'anomaly_score': np.random.uniform(0.55, 0.99, n),
        'dwell_hours':   np.random.exponential(6, n) + 2,
        'speed_kn':      np.random.exponential(1.2, n),
    })

df_ais_alerts = gen_ais_alerts()
print(f'AIS anomalous vessels loaded: {len(df_ais_alerts)}')

# ── Fire clusters ─────────────────────────────────────────────────────────────
firms_path = f'{PROCESSED}/firms_clusters.csv'
if os.path.exists(firms_path):
    df_fire = pd.read_csv(firms_path)
else:
    np.random.seed(13)
    centers = [(4.95, 6.85), (4.45, 6.10), (4.62, 6.43), (5.02, 6.70), (4.78, 6.55)]
    df_fire = pd.DataFrame({
        'cluster_id':   range(len(centers)),
        'centroid_lat': [c[0] for c in centers],
        'centroid_lon': [c[1] for c in centers],
        'frp_sum_mw':   np.random.uniform(50, 500, len(centers)),
        'risk_score':   np.random.uniform(30, 95,  len(centers)),
        'likely_illegal': [True, True, False, True, False],
    })
    print(f'⚠️  Using synthetic fire clusters: {len(df_fire)}')
else:
    print(f'Fire clusters loaded: {len(df_fire)}')


## 4.4 — Spatial join: assign detections to grid cells

In [ ]:
def score_cell_sar(cell_geom, df_blobs):
    """Sum spill probability scores within this cell."""
    if 'lat' not in df_blobs.columns:
        return 0.0
    inside = df_blobs[
        (df_blobs['lon'] >= cell_geom.bounds[0]) & (df_blobs['lon'] < cell_geom.bounds[2]) &
        (df_blobs['lat'] >= cell_geom.bounds[1]) & (df_blobs['lat'] < cell_geom.bounds[3])
    ]
    if inside.empty: return 0.0
    prob_col = 'spill_prob' if 'spill_prob' in inside.columns else pd.Series([0.7]*len(inside))
    return float(inside['spill_prob'].sum() if 'spill_prob' in inside.columns else len(inside)*0.7)

def score_cell_ais(cell_geom, df_ais):
    inside = df_ais[
        (df_ais['lon'] >= cell_geom.bounds[0]) & (df_ais['lon'] < cell_geom.bounds[2]) &
        (df_ais['lat'] >= cell_geom.bounds[1]) & (df_ais['lat'] < cell_geom.bounds[3])
    ]
    if inside.empty: return 0.0
    return float(inside['anomaly_score'].sum())

def score_cell_fire(cell_geom, df_fire, lat_col='centroid_lat', lon_col='centroid_lon'):
    inside = df_fire[
        (df_fire[lon_col] >= cell_geom.bounds[0]) & (df_fire[lon_col] < cell_geom.bounds[2]) &
        (df_fire[lat_col] >= cell_geom.bounds[1]) & (df_fire[lat_col] < cell_geom.bounds[3])
    ]
    if inside.empty: return 0.0
    return float(inside['risk_score'].sum() if 'risk_score' in inside.columns else len(inside)*40)

print('Computing per-cell risk scores...')
gdf_grid['sar_score']  = gdf_grid['geometry'].apply(lambda g: score_cell_sar(g,  df_blobs))
gdf_grid['ais_score']  = gdf_grid['geometry'].apply(lambda g: score_cell_ais(g,  df_ais_alerts))
gdf_grid['fire_score'] = gdf_grid['geometry'].apply(lambda g: score_cell_fire(g, df_fire))

# Normalise each score to 0-100
for col in ['sar_score','ais_score','fire_score']:
    mx = gdf_grid[col].max()
    gdf_grid[col+'_norm'] = (gdf_grid[col] / mx * 100).fillna(0) if mx > 0 else 0.0

# Weighted fusion: SAR spill 40%, AIS vessel 35%, Fire 25%
WEIGHTS = {'sar': 0.40, 'ais': 0.35, 'fire': 0.25}
gdf_grid['risk_score'] = (
    gdf_grid['sar_score_norm']  * WEIGHTS['sar'] +
    gdf_grid['ais_score_norm']  * WEIGHTS['ais'] +
    gdf_grid['fire_score_norm'] * WEIGHTS['fire']
)
gdf_grid['risk_score'] = gdf_grid['risk_score'].round(1)

# Alert level classification
def alert_level(score):
    if   score >= 65: return 'CRITICAL'
    elif score >= 40: return 'HIGH'
    elif score >= 20: return 'MEDIUM'
    else:             return 'LOW'

gdf_grid['alert_level'] = gdf_grid['risk_score'].apply(alert_level)

print('✅ Risk scores computed.')
print()
print(gdf_grid['alert_level'].value_counts().to_string())
print()
print('Top 10 highest-risk cells:')
top10 = gdf_grid.sort_values('risk_score', ascending=False).head(10)
print(top10[['cell_id','centroid_lat','centroid_lon','sar_score_norm',
             'ais_score_norm','fire_score_norm','risk_score','alert_level']].to_string(index=False))


## 4.5 — Visualise the risk grid (static)

In [ ]:
ALERT_COLORS = {'LOW':'#b7e4c7','MEDIUM':'#f9c74f','HIGH':'#f3722c','CRITICAL':'#d62828'}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Module 4 — Niger Delta Pipeline Risk Grid', fontsize=14)

for ax, score_col, title in [
    (axes[0], 'risk_score',       'Combined risk score (0–100)'),
    (axes[1], 'alert_level',      'Alert level classification'),
]:
    if score_col == 'risk_score':
        gdf_grid.plot(column='risk_score', cmap='RdYlGn_r', ax=ax,
                      legend=True, vmin=0, vmax=100,
                      legend_kwds={'label':'Risk score','shrink':0.6})
    else:
        for level, color in ALERT_COLORS.items():
            gdf_grid[gdf_grid['alert_level']==level].plot(ax=ax, color=color, edgecolor='white', linewidth=0.3)
        patches = [Patch(color=c, label=l) for l,c in ALERT_COLORS.items()]
        ax.legend(handles=patches, loc='lower right', fontsize=9)

    # Overlay detections
    if 'lat' in df_blobs.columns:
        ax.scatter(df_blobs['lon'], df_blobs['lat'], s=40, c='#023e8a', marker='o',
                   zorder=5, label='SAR spill blob', alpha=0.8)
    ax.scatter(df_ais_alerts['lon'], df_ais_alerts['lat'], s=30, c='purple', marker='^',
               zorder=5, label='AIS anomaly', alpha=0.7)
    ax.scatter(df_fire['centroid_lon'], df_fire['centroid_lat'], s=80, c='#e63946',
               marker='*', zorder=5, label='Fire cluster', alpha=0.9)
    ax.set_xlim(AOI_BOUNDS[0]-0.05, AOI_BOUNDS[2]+0.05)
    ax.set_ylim(AOI_BOUNDS[1]-0.05, AOI_BOUNDS[3]+0.05)
    ax.set_title(title); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.legend(fontsize=8)

plt.tight_layout()
fp = f'{OUTPUTS}/module4_risk_grid.png'
plt.savefig(fp, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Risk grid figure saved → {fp}')


## 4.6 — Export risk data & generate alert report

In [ ]:
# Save risk grid as GeoJSON (for Folium in Module 5)
grid_geojson_path = f'{PROCESSED}/risk_grid.geojson'
gdf_grid.drop(columns=['geometry']).to_csv(f'{PROCESSED}/risk_grid.csv', index=False)
gdf_grid.to_file(grid_geojson_path, driver='GeoJSON')
print(f'✅ Risk grid saved → {grid_geojson_path}')

# Alert report
n_critical = (gdf_grid['alert_level']=='CRITICAL').sum()
n_high     = (gdf_grid['alert_level']=='HIGH').sum()
n_medium   = (gdf_grid['alert_level']=='MEDIUM').sum()
n_low      = (gdf_grid['alert_level']=='LOW').sum()

report = {
    'generated': pd.Timestamp.now().isoformat(),
    'grid_cells': len(gdf_grid),
    'alert_counts': {'CRITICAL':int(n_critical),'HIGH':int(n_high),
                     'MEDIUM':int(n_medium),'LOW':int(n_low)},
    'top_cells': top10[['cell_id','centroid_lat','centroid_lon','risk_score','alert_level']]
                    .to_dict(orient='records'),
    'weights': WEIGHTS,
}
with open(f'{PROCESSED}/alert_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print()
print('='*52)
print('MODULE 4 — FUSION & RISK SCORING COMPLETE')
print('='*52)
print(f'  CRITICAL alerts : {n_critical} cells')
print(f'  HIGH alerts     : {n_high} cells')
print(f'  MEDIUM alerts   : {n_medium} cells')
print(f'  LOW             : {n_low} cells')
print()
print('Next → Module 5: Interactive dashboard & live demo')
